# Modelo multilingüe: embeddings de reportes → etiquetas de rodilla

**Flujo:** cargar datos → limpiar reportes → explorar → embeddings con RoBERTa biomédico-clínico → evaluar con Leave-One-Out (AUC por etiqueta) → guardar resultados.

**Índice**
1. Instalación e imports
2. Configuración
3. Carga de datos
4. Limpieza y normalización de reportes
5. Exploración del dataset
6. Modelo y tokenizer
7. Análisis de longitud de tokens
8. Funciones de embeddings
9. Generación de embeddings
10. Evaluación (Leave-One-Out)
11. Exportar resultados

## 1. Instalación e imports

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece langdetect

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel
from huggingface_hub import login

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import LeaveOneOut

from langdetect import detect, DetectorFactory

from google.colab import files, drive, userdata

## 2. Configuración

In [ ]:
# RoBERTa biomédico-clínico (preentrenado en español).
# Alternativa multilingüe para comparar: "xlm-roberta-base"
#MODEL_NAME = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 512

DATA_PATH   = "/content/train.csv"
OUTPUT_PATH = "/content/drive/MyDrive/train_with_language.csv"

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

Dispositivo: cpu


## 3. Carga de datos

In [ ]:
# Subir train.csv a Colab (solo la primera vez)
uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Filas totales      :", len(df))
print("Número de etiquetas:", len(LABELS))

Filas totales      : 4407
Número de etiquetas: 12


## 4. Limpieza y normalización de reportes

Antes de tokenizar se corrigen los textos. **El orden importa**:
1. **Arreglos específicos** (`limpiar_extra`): `∆` (signo matemático) → `Δ` (griega) y restos de RTF (`\'e0` → `à`, `\~` → espacio).
2. **Normalización NFKC**: `µ` (micro, U+00B5) → `μ` (griega, U+03BC), NBSP → espacio, etc.

Los escapes RTF se decodifican *antes* de NFKC: algunos producen caracteres (por ejemplo `\'b5` = `µ`) que NFKC todavía debe normalizar.

In [ ]:
DELTA_MAT, DELTA = "\u2206", "\u0394"   # ∆ "incremento" vs Δ griega: se ven iguales, NFKC no los une

# RTF (formato de Word) dejó escapes en algunos reportes holandeses:
#   \'e0 = byte 0xE0 en Windows-1252 = "à"  ("graad 2 \'e0 3" = "grado 2 a 3"),  \'b0 = "°"
#   \~   = espacio de no separación de RTF  ("12\~mm" = "12 mm")
PATRON_RTF_HEX = re.compile(r"\\'([0-9a-fA-F]{2})")


def _decodificar_hex_rtf(m):
    try:
        return bytes.fromhex(m.group(1)).decode("cp1252")
    except UnicodeDecodeError:          # bytes sin definición en cp1252 (0x81, 0x8d, ...)
        return m.group(0)


def limpiar_extra(texto):
    texto = texto.replace(DELTA_MAT, DELTA)
    texto = PATRON_RTF_HEX.sub(_decodificar_hex_rtf, texto)
    return texto.replace("\\~", " ")


def limpiar_reporte(texto):
    """Pipeline completo: arreglos RTF/∆ y luego normalización NFKC."""
    if pd.isna(texto):
        return texto
    texto = limpiar_extra(str(texto))
    return unicodedata.normalize("NFKC", texto)

In [ ]:
df["Report_norm"] = df["Report"].apply(limpiar_reporte)

cambiados = (df["Report"].notna() & (df["Report"] != df["Report_norm"])).sum()
print(f"Reportes modificados por la limpieza: {cambiados} de {df['Report'].notna().sum()}")

Reportes modificados por la limpieza: 438 de 4407


### 4.1 Verificación: ejemplos antes / después

In [ ]:
from difflib import SequenceMatcher


def mostrar_cambios(antes, despues, contexto=25):
    """Imprime los fragmentos que difieren entre dos textos."""
    sm = SequenceMatcher(None, antes, despues, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "equal":
            print(f"  {antes[max(i1 - contexto, 0):i2 + contexto]!r}")
            print(f"  → {despues[max(j1 - contexto, 0):j2 + contexto]!r}")


mask = df["Report"].notna() & (df["Report"] != df["Report_norm"])
for idx in df[mask].head(5).index:
    print(f"Fila {idx}:")
    mostrar_cambios(df.at[idx, "Report"], df.at[idx, "Report_norm"])

Fila 13:
  ' ΤΕΧΝΙΚΗ Η εξέταση έγινε µε ακολουθίες παλµών Τ1, Τ'
  → ' ΤΕΧΝΙΚΗ Η εξέταση έγινε με ακολουθίες παλμών Τ1, Τ'
  'η έγινε µε ακολουθίες παλµών Τ1, Τ2 me2d και Pd fs.'
  → 'η έγινε με ακολουθίες παλμών Τ1, Τ2 me2d και Pd fs.'
  'συλλογή υγρού. Ο οστικός µυελός ελέγχεται εντός του'
  → 'συλλογή υγρού. Ο οστικός μυελός ελέγχεται εντός του'
  'ού. Χωρίς παθολογικά ευρήµατα ελέγχεται ο αρθρικός '
  → 'ού. Χωρίς παθολογικά ευρήματα ελέγχεται ο αρθρικός '
  'ρικός χόνδρος. Χωρίς ευρήµατα δυσλειτουργίας του εκ'
  → 'ρικός χόνδρος. Χωρίς ευρήματα δυσλειτουργίας του εκ'
  'ειτουργίας του εκτατικού µηχανισµού. Εντός του φυσι'
  → 'ειτουργίας του εκτατικού μηχανισμού. Εντός του φυσι'
  'ίας του εκτατικού µηχανισµού. Εντός του φυσιολογικο'
  → 'ίας του εκτατικού μηχανισμού. Εντός του φυσιολογικο'
  'σιολογικού ελέγχονται οι µηνίσκοι, οι πλάγιοι και ο'
  → 'σιολογικού ελέγχονται οι μηνίσκοι, οι πλάγιοι και ο'
  'ιοι και οι χιαστοί σύνδεσµοι.'
  → 'ιοι και οι χιαστοί σύνδεσμοι.'
Fila 15:


In [ ]:
# Comprobar que no quedan restos de RTF, ∆ ni µ
resto = df["Report_norm"].dropna()
print("Con escapes RTF restantes:", resto.str.contains(r"\\'[0-9a-fA-F]{2}|\\~", regex=True).sum())
print("Con ∆ (U+2206) restante  :", resto.str.contains(DELTA_MAT).sum())
print("Con µ (U+00B5) restante  :", resto.str.contains("\u00b5").sum())

Con escapes RTF restantes: 0
Con ∆ (U+2206) restante  : 0
Con µ (U+00B5) restante  : 0


### 4.2 Idioma de cada reporte

El modelo biomédico-clínico fue preentrenado en **español**, pero el dataset incluye reportes en otros idiomas (p. ej. holandés). Detectar el idioma permite ver cuánto texto cae fuera del dominio del modelo.

In [ ]:
DetectorFactory.seed = 0   # resultados reproducibles


def detectar_idioma(texto):
    try:
        return detect(texto)
    except Exception:       # texto vacío o demasiado corto
        return "unknown"


df["Language"] = df["Report_norm"].fillna("").apply(detectar_idioma)
df["Language"].value_counts()

,count
Language,
en,1736
es,682
tr,546
hr,406
el,321
de,262
bg,220
nl,153
fr,81


## 5. Exploración del dataset

### 5.1 Resumen por etiqueta

In [ ]:
summary_df = pd.DataFrame([
    {
        "Label": col,
        "Positive (1)": (df[col] == 1).sum(),
        "Negative (0)": (df[col] == 0).sum(),
        "Missing (NaN)": df[col].isna().sum(),
        "Total": len(df),
    }
    for col in LABELS
])

display(summary_df)

,Label,Positive (1),Negative (0),Missing (NaN),Total
0,ACL,24,34,4349,4407
1,MCL,9,49,4349,4407
2,Medial Meniscus,26,32,4349,4407
3,Lateral Meniscus,23,35,4349,4407
4,Medial OA,15,43,4349,4407
5,Lateral OA,11,47,4349,4407
6,PF OA,21,37,4349,4407
7,Effusion,35,23,4349,4407
8,Synovitis,27,31,4349,4407
9,Baker's,12,46,4349,4407


### 5.2 Valores únicos por etiqueta

In [ ]:
for col in LABELS:
    print(f"{col:20s} → {sorted(df[col].dropna().unique())}")

ACL                  → [np.float64(0.0), np.float64(1.0)]
MCL                  → [np.float64(0.0), np.float64(1.0)]
Medial Meniscus      → [np.float64(0.0), np.float64(1.0)]
Lateral Meniscus     → [np.float64(0.0), np.float64(1.0)]
Medial OA            → [np.float64(0.0), np.float64(1.0)]
Lateral OA           → [np.float64(0.0), np.float64(1.0)]
PF OA                → [np.float64(0.0), np.float64(1.0)]
Effusion             → [np.float64(0.0), np.float64(1.0)]
Synovitis            → [np.float64(0.0), np.float64(1.0)]
Baker's              → [np.float64(0.0), np.float64(1.0)]
Contusion            → [np.float64(0.0), np.float64(1.0)]
Fracture             → [np.float64(0.0), np.float64(1.0)]


### 5.3 Reportes de texto

In [ ]:
print("Reports con texto:", df["Report"].notna().sum())
print("Reports vacíos   :", df["Report"].isna().sum())

display(
    df[["StudyInstanceUID", "Report"]]
    .dropna(subset=["Report"])
    .head(5)
)

Reports con texto: 4407
Reports vacíos   : 0


,StudyInstanceUID,Report
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no..."
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...


### 5.4 Estudios etiquetados

In [ ]:
# Estudios con al menos una etiqueta
labeled_any = df[df[LABELS].notna().any(axis=1)]
print("Estudios con alguna etiqueta:", len(labeled_any))

# Subconjunto de trabajo: estudios con etiqueta ACL
# (el resto de etiquetas aparece junto con ACL)
df_labeled = df[df["ACL"].notna()].copy()
print("Estudios etiquetados (ACL) :", len(df_labeled))

Estudios con alguna etiqueta: 58
Estudios etiquetados (ACL) : 58


## 6. Modelo y tokenizer

In [ ]:
# Opcional: autenticación en Hugging Face (el modelo es público, solo evita límites de descarga)
HF_TOKEN = userdata.get("HF_TOKEN")
print("Token encontrado:", HF_TOKEN is not None)

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Hugging Face autenticado correctamente")

Token encontrado: True
Hugging Face autenticado correctamente


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("Modelo cargado")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo cargado


## 7. Análisis de longitud de tokens

In [ ]:
texts = df["Report_norm"].fillna("").astype(str)

token_lengths = texts.apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

print("Tokens mínimos :", token_lengths.min())
print("Tokens máximos :", token_lengths.max())
print("Tokens promedio:", round(token_lengths.mean(), 1))
print("Mediana        :", token_lengths.median())
print()

for limit in [128, 256, 384, 512]:
    print(f"Más de {limit:3d} tokens: {(token_lengths > limit).mean() * 100:.2f}%")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (534 > 512). Running this sequence through the model will result in indexing errors


Tokens mínimos : 19
Tokens máximos : 1238
Tokens promedio: 310.3
Mediana        : 281.0

Más de 128 tokens: 82.41%
Más de 256 tokens: 54.82%
Más de 384 tokens: 29.41%
Más de 512 tokens: 14.00%


## 8. Funciones de embeddings

Una sola función con dos tipos de pooling:
- `"cls"`: embedding del token `[CLS]` (primer token).
- `"mean"`: promedio de todos los tokens, ponderado por la máscara de atención.

In [ ]:
@torch.no_grad()
def report_to_embedding(report, pooling="mean"):
    """Convierte un reporte en un vector de embedding (768 dims para xlm-roberta-base)."""
    inputs = tokenizer(
        str(report),
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    hidden = model(**inputs).last_hidden_state

    if pooling == "cls":
        emb = hidden[:, 0, :]
    elif pooling == "mean":
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    else:
        raise ValueError(f"pooling desconocido: {pooling}")

    return emb.cpu().numpy().flatten()


def build_embeddings(reports, pooling="mean"):
    """Calcula los embeddings de una serie de reportes."""
    return np.array([
        report_to_embedding(r, pooling) for r in tqdm(reports, desc=f"Embeddings ({pooling})")
    ])

## 9. Generación de embeddings

In [ ]:
X_cls  = build_embeddings(df_labeled["Report_norm"], pooling="cls")
X_mean = build_embeddings(df_labeled["Report_norm"], pooling="mean")
y = df_labeled[LABELS].values

print("X_cls :", X_cls.shape)
print("X_mean:", X_mean.shape)
print("y     :", y.shape)

Embeddings (cls):   0%|          | 0/58 [00:00<?, ?it/s]

Embeddings (mean):   0%|          | 0/58 [00:00<?, ?it/s]

X_cls : (58, 768)
X_mean: (58, 768)
y     : (58, 12)


In [ ]:
np.save("X_text_embeddings_cls.npy",  X_cls)
np.save("X_text_embeddings_mean.npy", X_mean)
np.save("y_labels.npy", y)

## 10. Evaluación con Leave-One-Out

Regresión logística (`class_weight="balanced"`) por etiqueta, evaluada con AUC.

In [ ]:
def loo_auc(X, y_label):
    """AUC con Leave-One-Out para una etiqueta binaria."""
    y_true, y_prob = [], []

    for train_idx, test_idx in LeaveOneOut().split(X):
        clf = LogisticRegression(max_iter=5000, class_weight="balanced")
        clf.fit(X[train_idx], y_label[train_idx])

        y_prob.append(clf.predict_proba(X[test_idx])[0, 1])
        y_true.append(y_label[test_idx][0])

    return roc_auc_score(y_true, y_prob)


def evaluate_all_labels(X, name=""):
    """AUC por etiqueta, ordenado de mayor a menor."""
    results = {
        label: loo_auc(X, df_labeled[label].astype(int).values)
        for label in tqdm(LABELS, desc=f"LOO {name}")
    }
    return pd.Series(results, name=f"AUC ({name})").sort_values(ascending=False)

In [ ]:
auc_cls = evaluate_all_labels(X_cls, "CLS")
auc_cls.round(4)

LOO CLS:   0%|          | 0/12 [00:00<?, ?it/s]

,AUC (CLS)
Medial Meniscus,0.6719
Lateral OA,0.5783
Contusion,0.5479
Lateral Meniscus,0.5317
Synovitis,0.5257
ACL,0.4963
Medial OA,0.4760
Effusion,0.4584
PF OA,0.4363
Fracture,0.4333


In [ ]:
auc_mean = evaluate_all_labels(X_mean, "Mean")
auc_mean.round(4)

LOO Mean:   0%|          | 0/12 [00:00<?, ?it/s]

,AUC (Mean)
Medial Meniscus,0.6935
Medial OA,0.6233
Lateral Meniscus,0.6025
Lateral OA,0.5919
ACL,0.5809
Fracture,0.5333
Contusion,0.5223
Effusion,0.5093
Synovitis,0.4683
PF OA,0.4569


### Comparación CLS vs Mean pooling

In [ ]:
comparison = pd.concat([auc_cls, auc_mean], axis=1).round(4)
comparison["Mejor"] = comparison.idxmax(axis=1)
display(comparison)

print("AUC promedio:")
print(comparison.iloc[:, :2].mean().round(4))

,AUC (CLS),AUC (Mean),Mejor
Medial Meniscus,0.6719,0.6935,AUC (Mean)
Lateral OA,0.5783,0.5919,AUC (Mean)
Contusion,0.5479,0.5223,AUC (CLS)
Lateral Meniscus,0.5317,0.6025,AUC (Mean)
Synovitis,0.5257,0.4683,AUC (CLS)
ACL,0.4963,0.5809,AUC (Mean)
Medial OA,0.4760,0.6233,AUC (Mean)
Effusion,0.4584,0.5093,AUC (Mean)
PF OA,0.4363,0.4569,AUC (Mean)
Fracture,0.4333,0.5333,AUC (Mean)


AUC promedio:
AUC (CLS)     0.4921
AUC (Mean)    0.5260
dtype: float64


## 11. Exportar resultados

In [ ]:
drive.mount("/content/drive")
df.to_csv(OUTPUT_PATH, index=False)
print("Guardado en:", OUTPUT_PATH)

In [ ]:
# Opcional: descargar a tu PC
files.download(OUTPUT_PATH)